In [ ]:
import cdfmm
import numpy as np
import time

In [ ]:
# ==================================================================
# Physical problem setup
# ==================================================================

N = 40000                  # Number of dipoles
NT = 10                    # Number of repeated field evaluations

L_nm = 100.0               # Side length of the simulation cube [nm]

mu0_Ms = 1.5               # Saturation magnetization expressed as mu0*Ms [T]

moment_scale_min = 0.5     # 0.5 -> half the reference cell moment
moment_scale_max = 1.5     # 1.5 -> 1.5 times the reference cell moment

# Keep geometry and moment generation independent.
#
# The moment RNG is reset to exactly the same seed before each backend loop.
# Therefore time step t uses exactly the same dipole moments for:
#
#   - FMM
#   - CPU direct
#   - CUDA direct
#
geometry_seed = 42
moment_seed = 12345

geometry_rng = np.random.default_rng(seed=geometry_seed)


In [ ]:
# ------------------------------------------------------------------
# Convert to SI units
# ------------------------------------------------------------------

mu0 = 4.0 * np.pi * 1e-7          # Vacuum permeability [T m / A]

L = L_nm * 1e-9                    # Simulation side length [m]

Ms = mu0_Ms / mu0                  # Saturation magnetization [A/m]


# ------------------------------------------------------------------
# Equivalent micromagnetic cell
# ------------------------------------------------------------------

# Imagine that the full volume L^3 were divided evenly between N
# uniformly magnetized cubic cells.
#
# Each cell would then have volume
#
#       V_cell = L^3 / N
#
cell_volume = L**3 / N             # [m^3]

# Equivalent cube side length
cell_size = cell_volume**(1.0 / 3.0)   # [m]

# A uniformly magnetized cell with magnetization Ms has dipole moment
#
#       m_ref = Ms * V_cell
#
# This defines "moment strength = 1".
#
moment_ref = Ms * cell_volume       # [A m^2]


# ==================================================================
# Random dipole positions
# ==================================================================

# Random positions distributed uniformly through the simulation cube:
#
#   [-L/2, L/2] x [-L/2, L/2] x [-L/2, L/2]
#
# The geometry is generated only once and remains fixed for all NT updates.
#
positions = geometry_rng.uniform(
    -L / 2,
    L / 2,
    size=(N, 3),
)


# ==================================================================
# Random dipole moment generator
# ==================================================================

def generate_random_moments(rng):
    """Generate one random dipole-moment state."""

    # Isotropically distributed random directions
    directions = rng.normal(size=(N, 3))
    directions /= np.linalg.norm(directions, axis=1)[:, None]

    # Random moment strengths between moment_scale_min and moment_scale_max.
    #
    # A strength of 1 corresponds to:
    #
    #       |m| = Ms * V_cell
    #
    moment_scales = rng.uniform(
        moment_scale_min,
        moment_scale_max,
        size=N,
    )

    return directions * (moment_ref * moment_scales)[:, None]


In [ ]:

# ==================================================================
# FMM options
# ==================================================================

options = cdfmm.UniformFmmOptions()
options.fixed_target_source_indices = np.arange(N, dtype=int).tolist()


# ------------------------------------------------------------------
# Expansion order
# ------------------------------------------------------------------

# Maximum total degree of the Cartesian multipole expansion.
#
# Available choices:
#   Any integer >= 0
#
# Higher order:
#   + Higher accuracy
#   - More coefficients and more computational work
#
options.expansion_order = 4


# ------------------------------------------------------------------
# Tree depth
# ------------------------------------------------------------------

# Maximum level of the uniform octree.
#
# Available choices:
#   Any integer >= 0
#
#   0 = root box only
#   1 = root + 8 child boxes
#   2 = root + two subdivision levels
#   ...
#
# Deeper trees:
#   + Fewer particles in each leaf / less direct near-field work
#   - More tree boxes and FMM translations
#
options.tree.max_level = 4


# ------------------------------------------------------------------
# Empty nodes
# ------------------------------------------------------------------

# Whether empty boxes should be included.
#
# Available choices:
#   True
#   False
#
# CURRENT IMPLEMENTATION:
#   All boxes are currently materialised regardless of this setting.
#
options.tree.include_empty_nodes = True


# ------------------------------------------------------------------
# Root box shape
# ------------------------------------------------------------------

# Whether the root box should be cubic.
#
# Available choices:
#   True
#   False
#
# CURRENT IMPLEMENTATION:
#   Only cubic root boxes are currently supported.
#
options.tree.cubic_root_box = True


# ------------------------------------------------------------------
# Root box centre
# ------------------------------------------------------------------

# Available choices:
#   None                    -> automatically determine from the positions
#   cdfmm.Vec3(x, y, z)    -> explicitly specify the centre
#
# Since the random positions were generated inside a cube centred at zero,
# we specify the centre explicitly here.
#
options.tree.root_centre = None
#options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)


# ------------------------------------------------------------------
# Root box half-width
# ------------------------------------------------------------------

# Available choices:
#   None           -> automatically determine from the positions
#   positive float -> explicitly specify the half-width
#
# The box has side length L, so its half-width is L/2.
#

options.tree.root_half_width = None
#options.tree.root_half_width = L / 2



# ------------------------------------------------------------------
# M2L implementation
# ------------------------------------------------------------------

# Available choices:
#
#   cdfmm.M2LBackend.Static
#       Uses precomputed/cached M2L translation matrices.
#       This is the normal optimized implementation.
#
#   cdfmm.M2LBackend.Reference
#       Uses the reference M2L implementation.
#       Mainly intended for validation/testing.
#
options.m2l_backend = cdfmm.M2LBackend.Static


# ------------------------------------------------------------------
# Static matrix multiplication backend
# ------------------------------------------------------------------

# Available choices:
#
#   cdfmm.StaticMatrixBackend.PORTABLE
#       Portable CPU implementation.
#
#   cdfmm.StaticMatrixBackend.ONE_MKL
#       Uses Intel oneMKL for grouped M2L matrix multiplication.
#       Requires the library to have been built with oneMKL support.
#
options.static_matrix_backend = cdfmm.StaticMatrixBackend.ONE_MKL


# ------------------------------------------------------------------
# Complete FMM execution backend
# ------------------------------------------------------------------

# Available choices:
#
#   cdfmm.ExecutionBackend.AUTO
#       Automatically selects a backend.
#       Currently resolves conservatively to CPU_STATIC.
#
#   cdfmm.ExecutionBackend.CPU_REFERENCE
#       Reference CPU FMM implementation.
#       Mainly for validation/testing.
#
#   cdfmm.ExecutionBackend.CPU_STATIC
#       Optimized static CPU FMM.
#
#   cdfmm.ExecutionBackend.CUDA_PARTIAL
#       P2M/M2M/L2L/L2P on CPU.
#       M2L and P2P on GPU.
#
#   cdfmm.ExecutionBackend.CUDA_FULL
#       Complete FMM evaluation on the GPU.
#
# Compatibility aliases for CUDA_PARTIAL also exist:
#   CUDA_M2L_P2P
#   CUDA_M2L
#   CUDA_M2L_STATIC_P2P
#
options.backend = cdfmm.ExecutionBackend.CUDA_FULL

In [ ]:
# ==================================================================
# Create FMM
# ==================================================================

# We evaluate the field at the dipole positions themselves, so positions
# are both the source and target coordinates.

start = time.perf_counter()

fmm = cdfmm.UniformFmm(
    positions,
    positions,
    options,
)

fmm_setup_time = time.perf_counter() - start

In [ ]:
# ==================================================================
# FMM evaluation
# ==================================================================

# Target i is the same physical particle as source i.
# This tells the FMM to exclude self-interaction.
self_indices = np.arange(N)

# Store the fields from every update so that accuracy can later be compared
# update-by-update against the direct reference.
#
# Memory usage is approximately:
#
#       NT * N * 3 * 8 bytes
#
H_fmm_all = np.empty((NT, N, 3), dtype=float)

# Reset the moment RNG before the FMM loop.
moment_rng = np.random.default_rng(seed=moment_seed)

start = time.perf_counter()

for t in range(NT):
    moments = generate_random_moments(moment_rng)

    result = fmm.evaluate(
        moments,
        output="field",
        target_source_indices=self_indices,
    )

    H_fmm_all[t] = result["H"]

fmm_evaluation_time = time.perf_counter() - start

# Keep the final field available under the same name as in the single-update
# notebook for convenient inspection.
H_fmm = H_fmm_all[-1]


In [ ]:
# ==================================================================
# Direct evaluation
# ==================================================================

# The direct method evaluates every target against every source:
#
#       O(N^2)
#
# CPU direct:
#   direct_p2p_reference() is called normally NT times.
#
# CUDA direct:
#   CudaDirectPlan is constructed ONCE from the fixed source/target geometry.
#   The geometry therefore stays resident on the GPU.
#
#   Each repeated evaluation then transfers only the new moments to the device,
#   runs the direct P2P kernel, and transfers the resulting field back.


# ------------------------------------------------------------------
# CPU direct evaluation
# ------------------------------------------------------------------

H_direct_cpu_all = np.empty((NT, N, 3), dtype=float)

# Reset to the SAME moment seed used by the FMM loop.
moment_rng = np.random.default_rng(seed=moment_seed)

start = time.perf_counter()

for t in range(NT):
    moments = generate_random_moments(moment_rng)

    result_direct_cpu = cdfmm.direct_p2p_reference(
        positions,
        positions,
        moments,
        output="field",
        target_source_indices=self_indices,
    )

    H_direct_cpu_all[t] = result_direct_cpu["H"]

direct_cpu_time = time.perf_counter() - start

H_direct_cpu = H_direct_cpu_all[-1]


# ------------------------------------------------------------------
# GPU direct evaluation
# ------------------------------------------------------------------

H_direct_gpu_all = None
H_direct_gpu = None
direct_gpu_setup_time = None
direct_gpu_time = None

if cdfmm.cuda_direct_available():

    # Create the persistent direct plan once.
    #
    # Fixed geometry and the self-identity map are uploaded during setup.
    start = time.perf_counter()

    direct_gpu = cdfmm.CudaDirectPlan(
        positions,
        positions,
        target_source_indices=self_indices,
    )

    direct_gpu_setup_time = time.perf_counter() - start

    H_direct_gpu_all = np.empty((NT, N, 3), dtype=float)

    # Reset to the SAME moment seed used by the FMM and CPU-direct loops.
    moment_rng = np.random.default_rng(seed=moment_seed)

    start = time.perf_counter()

    for t in range(NT):
        moments = generate_random_moments(moment_rng)

        result_direct_gpu = direct_gpu.evaluate(
            moments,
            output="field",
        )

        H_direct_gpu_all[t] = result_direct_gpu["H"]

    direct_gpu_time = time.perf_counter() - start

    H_direct_gpu = H_direct_gpu_all[-1]

else:
    print("CUDA direct evaluation is not available.")


In [ ]:
# ==================================================================
# Accuracy comparison
# ==================================================================

# Use the CPU direct calculation as the reference solution.
#
# We compare all NT updates together:
#
#   1. FMM        vs CPU direct
#   2. GPU direct vs CPU direct   (when CUDA direct is available)
#
# The arrays are reshaped from (NT, N, 3) -> (NT*N, 3), so every field
# vector from every update contributes to the reported error metrics.


def field_error_metrics(H, H_reference):
    error = H - H_reference

    # Absolute vector error at each target
    absolute_error = np.linalg.norm(error, axis=1)

    # Magnitude of the reference field at each target
    reference_magnitude = np.linalg.norm(H_reference, axis=1)

    # Pointwise relative vector error.
    # The epsilon prevents division by zero for vanishing reference fields.
    relative_error = absolute_error / np.maximum(
        reference_magnitude,
        np.finfo(float).eps,
    )

    # Global relative L2 error
    relative_l2_error = (
        np.linalg.norm(error)
        / np.linalg.norm(H_reference)
    )

    # RMS error over all field components
    rmse = np.sqrt(np.mean(error**2))

    return {
        "mean_absolute_error": np.mean(absolute_error),
        "max_absolute_error": np.max(absolute_error),
        "mean_relative_error": np.mean(relative_error),
        "max_relative_error": np.max(relative_error),
        "relative_l2_error": relative_l2_error,
        "rmse": rmse,
    }


H_fmm_flat = H_fmm_all.reshape(-1, 3)
H_direct_cpu_flat = H_direct_cpu_all.reshape(-1, 3)

# FMM accuracy relative to the CPU direct reference
fmm_error = field_error_metrics(
    H_fmm_flat,
    H_direct_cpu_flat,
)


# GPU direct accuracy relative to the CPU direct reference
gpu_direct_error = None

if H_direct_gpu_all is not None:
    H_direct_gpu_flat = H_direct_gpu_all.reshape(-1, 3)

    gpu_direct_error = field_error_metrics(
        H_direct_gpu_flat,
        H_direct_cpu_flat,
    )


In [ ]:
# ==================================================================
# Results
# ==================================================================

fmm_total_time = fmm_setup_time + fmm_evaluation_time

direct_gpu_total_time = None
if direct_gpu_time is not None:
    direct_gpu_total_time = direct_gpu_setup_time + direct_gpu_time


# ------------------------------------------------------------------
# Problem information
# ------------------------------------------------------------------

print()
print("Physical problem")
print("----------------")
print(f"N:                        {N}")
print(f"NT:                       {NT}")
print(f"Simulation side length:   {L_nm:.1f} nm")
print(f"mu0 Ms:                   {mu0_Ms:.3f} T")
print(f"Ms:                       {Ms:.6e} A/m")
print(f"Equivalent cell size:     {cell_size * 1e9:.3f} nm")
print(f"Reference dipole moment:  {moment_ref:.6e} A m^2")
print(
    f"Moment scale range:        "
    f"{moment_scale_min:.2f} - {moment_scale_max:.2f}"
)
print(f"Geometry seed:             {geometry_seed}")
print(f"Moment seed:               {moment_seed}")

print()
print("FMM parameters")
print("--------------")
print(f"Expansion order:           {options.expansion_order}")
print(f"Tree depth:                {options.tree.max_level}")


# ------------------------------------------------------------------
# Timing
# ------------------------------------------------------------------

print()
print("Timing")
print("------")
print(f"FMM setup:                 {fmm_setup_time:.6f} s")
print(f"FMM {NT} evaluations:      {fmm_evaluation_time:.6f} s")
print(f"FMM mean / evaluation:     {fmm_evaluation_time / NT:.6f} s")
print(f"FMM setup + evaluations:   {fmm_total_time:.6f} s")

print()
print(f"CPU direct {NT} evals:     {direct_cpu_time:.6f} s")
print(f"CPU direct mean / eval:    {direct_cpu_time / NT:.6f} s")

if direct_gpu_time is not None:
    print()
    print(f"GPU direct setup:          {direct_gpu_setup_time:.6f} s")
    print(f"GPU direct {NT} evals:     {direct_gpu_time:.6f} s")
    print(f"GPU direct mean / eval:    {direct_gpu_time / NT:.6f} s")
    print(f"GPU setup + evaluations:   {direct_gpu_total_time:.6f} s")


# ------------------------------------------------------------------
# Speedup
# ------------------------------------------------------------------

print()
print("Speedup")
print("-------")

print(
    f"FMM eval vs CPU direct:    "
    f"{direct_cpu_time / fmm_evaluation_time:.2f}x"
)

print(
    f"FMM total vs CPU direct:   "
    f"{direct_cpu_time / fmm_total_time:.2f}x"
)

if direct_gpu_time is not None:
    print(
        f"GPU direct vs CPU direct:  "
        f"{direct_cpu_time / direct_gpu_time:.2f}x"
    )

    print(
        f"GPU direct total vs CPU:   "
        f"{direct_cpu_time / direct_gpu_total_time:.2f}x"
    )

    print(
        f"FMM eval vs GPU direct:    "
        f"{direct_gpu_time / fmm_evaluation_time:.2f}x"
    )

    print(
        f"FMM total vs GPU direct:   "
        f"{direct_gpu_time / fmm_total_time:.2f}x"
    )


# ------------------------------------------------------------------
# Accuracy: FMM vs CPU direct
# ------------------------------------------------------------------

print()
print(f"Accuracy over all {NT} updates: FMM vs CPU direct")
print("------------------------------------------------")
print(
    f"Relative L2 error:         "
    f"{fmm_error['relative_l2_error']:.6e}"
)
print(
    f"Mean relative error:       "
    f"{fmm_error['mean_relative_error']:.6e}"
)
print(
    f"Maximum relative error:    "
    f"{fmm_error['max_relative_error']:.6e}"
)
print(
    f"RMSE:                      "
    f"{fmm_error['rmse']:.6e} A/m"
)
print(
    f"Mean absolute error:       "
    f"{fmm_error['mean_absolute_error']:.6e} A/m"
)
print(
    f"Maximum absolute error:    "
    f"{fmm_error['max_absolute_error']:.6e} A/m"
)


# ------------------------------------------------------------------
# Accuracy: GPU direct vs CPU direct
# ------------------------------------------------------------------

if gpu_direct_error is not None:

    print()
    print(f"Accuracy over all {NT} updates: GPU direct vs CPU direct")
    print("-------------------------------------------------------")
    print(
        f"Relative L2 difference:    "
        f"{gpu_direct_error['relative_l2_error']:.6e}"
    )
    print(
        f"Mean relative difference:  "
        f"{gpu_direct_error['mean_relative_error']:.6e}"
    )
    print(
        f"Maximum relative diff.:    "
        f"{gpu_direct_error['max_relative_error']:.6e}"
    )
    print(
        f"RMSE:                      "
        f"{gpu_direct_error['rmse']:.6e} A/m"
    )
